# 06 — Modelado completo y búsqueda de hiperparámetros

Notebook de modelado y búsqueda de hiperparámetros sobre `features_baseline.parquet`.

**Reglas obligatorias**

- Target: `rhythm_label` (clasificación multiclase de arritmias intraoperatorias).
- `case_id` se usa como grupo para split y CV — sin leakage entre pacientes.
- `beat_type` está prohibido como predictor (bloqueado por `assert_no_forbidden_features`).
- Split 80/20 por `case_id` con cobertura de clases (`make_train_test_group_split_with_coverage`).
- CV interna por grupo (`StratifiedGroupKFold` cuando es viable; `GroupKFold` como fallback).
- El test se evalúa **una sola vez** al final, después de fijar hiperparámetros con CV en train.
- Métrica primaria: `f1_macro`. Complementarias: `precision_macro`, `recall_macro`, `accuracy`.

**Modelos comparados**

SVM (`LinearSVC`), Árbol de Decisión, Random Forest, XGBoost, MLP.

**Pre-requisito**

Haber ejecutado `04_windowing_and_feature_engineering.ipynb` para generar `data/processed/features_baseline.parquet`.

## 1. Setup

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from src import config
from src.evaluation import confusion_matrix_with_totals, per_class_report
from src.modeling import (
    assert_no_forbidden_features,
    make_train_test_group_split_with_coverage,
)
from src.search import (
    MODEL_REGISTRY,
    PRIMARY_SCORING,
    build_cv_splitter,
    evaluate_on_test,
    run_search_for_model,
)
from src.utils import get_logger, set_seed

set_seed(config.RANDOM_SEED)
logger = get_logger("nb06")
sns.set_theme(context="notebook", style="whitegrid")

## 2. Configuración del run

Ajusta `FAST_MODE` según el alcance:
- `FAST_MODE = True`  → rápido (N_ITER=5, N_SPLITS=3) — útil para depurar.
- `FAST_MODE = False` → completo (N_ITER=30, N_SPLITS=5) — resultados finales.

In [3]:
FAST_MODE = True

N_ITER   = 5  if FAST_MODE else 30
N_SPLITS = 3  if FAST_MODE else 5
TEST_SIZE = 0.2
N_JOBS   = -1

# Modelos a comparar (todos los comprometidos en el proyecto)
MODELS_TO_RUN = ["linear_svc", "decision_tree", "random_forest", "xgboost", "mlp"]

print(f"FAST_MODE : {FAST_MODE}")
print(f"N_ITER    : {N_ITER}")
print(f"N_SPLITS  : {N_SPLITS}")
print(f"TEST_SIZE : {TEST_SIZE}")
print(f"Modelos   : {MODELS_TO_RUN}")

FAST_MODE : True
N_ITER    : 5
N_SPLITS  : 3
TEST_SIZE : 0.2
Modelos   : ['linear_svc', 'decision_tree', 'random_forest', 'xgboost', 'mlp']


## 3. Diagnóstico inicial

Se imprime antes de entrenar para detectar problemas de carga, leakage o features incorrectas.

In [4]:
features_parquet = config.PROCESSED_DIR / "features_baseline.parquet"
features_csv     = config.PROCESSED_DIR / "features_baseline.csv"

print("=" * 70)
print("DIAGNÓSTICO INICIAL")
print("=" * 70)
print(f"Ruta parquet  : {features_parquet}")
print(f"Existe parquet: {features_parquet.exists()}")
print(f"Ruta CSV      : {features_csv}")
print(f"Existe CSV    : {features_csv.exists()}")
print("=" * 70)

DIAGNÓSTICO INICIAL
Ruta parquet  : C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\processed\features_baseline.parquet
Existe parquet: True
Ruta CSV      : C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\processed\features_baseline.csv
Existe CSV    : False


## 4. Carga robusta de datos

Intenta cargar `features_baseline.parquet`; si falla (pyarrow ausente, archivo corrupto), usa el CSV como fallback.

In [5]:
try:
    df_raw = pd.read_parquet(features_parquet)
    load_source = f"parquet ({features_parquet.name})"
except Exception as e_parquet:
    print(f"[AVISO] No se pudo leer parquet: {e_parquet}")
    print(f"Intentando CSV: {features_csv}")
    df_raw = pd.read_csv(features_csv)
    load_source = f"CSV ({features_csv.name})"

# Limpiar filas con label inválido (NaN real o string 'nan')
n_raw = len(df_raw)
df = df_raw.dropna(subset=[config.TARGET_COLUMN])
mask_string_nan = (
    df[config.TARGET_COLUMN]
    .astype(str).str.strip().str.lower()
    .isin({"nan", "none", ""})
)
df = df.loc[~mask_string_nan].copy()
n_clean = len(df)

print(f"Fuente          : {load_source}")
print(f"Filas cargadas  : {n_raw:,}  |  Con label válido: {n_clean:,}")
print(f"Shape           : {df.shape}")
print(f"\nColumnas ({len(df.columns)}):")
print(list(df.columns))
print(f"\nConteo de {config.TARGET_COLUMN}:")
print(df[config.TARGET_COLUMN].value_counts().sort_index())
print(f"\ncase_id únicos: {df[config.CASE_ID_COLUMN].nunique()}")

Fuente          : parquet (features_baseline.parquet)
Filas cargadas  : 638,690  |  Con label válido: 638,690
Shape           : (638690, 39)

Columnas (39):
['case_id', 'window_id', 'beat_index', 'start_sample', 'end_sample', 'start_time', 'end_time', 'rhythm_label', 'original_nan_pct', 'max_nan_gap_seconds', 'was_interpolated', 'quality_status', 'quality_reason', 'mean', 'std', 'var', 'min', 'max', 'range', 'median', 'p25', 'p75', 'iqr', 'skew', 'kurtosis', 'energy', 'zero_crossing_rate', 'abs_mean', 'rr_prev', 'rr_next', 'rr_mean_local', 'rr_ratio', 'case_rr_count', 'case_rr_mean', 'case_rr_std', 'case_rr_min', 'case_rr_max', 'case_rr_rmssd', 'case_rr_pnn50']

Conteo de rhythm_label:
rhythm_label
AFIB/AFL                        158473
AVB                               4193
N                               391877
Patterned Atrial Ectopy          19946
Patterned Ventricular Ectopy     23902
SND                              22224
SVTA                              6396
Unclassifiable     

## 5. Selección segura de features

Reutiliza la lógica del notebook 05: excluye metadatos, columnas de auditoría de calidad y cualquier
columna prohibida por `config.FORBIDDEN_FEATURE_COLUMNS`. Luego selecciona únicamente columnas numéricas.
Se verifica que `rhythm_label`, `case_id` y `beat_type` no entren como predictores.

In [6]:
# Columnas a excluir explícitamente (además de FORBIDDEN_FEATURE_COLUMNS)
EXTRA_NON_FEATURES = {
    "beat_index", "start_sample", "end_sample",
    "window_id",  "start_time",  "end_time",
    "quality_status", "quality_reason",
    "original_nan_pct", "max_nan_gap_seconds", "was_interpolated",
    "window_seconds",
    config.BEAT_TIME_COLUMN,
}
non_feature_set = set(config.FORBIDDEN_FEATURE_COLUMNS) | EXTRA_NON_FEATURES

# Candidatas → solo numéricas
candidate_cols = [c for c in df.columns if c not in non_feature_set]
feature_cols = (
    df[candidate_cols]
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

# Validaciones metodológicas (deben pasar sin AssertionError)
assert_no_forbidden_features(feature_cols)
assert config.TARGET_COLUMN not in feature_cols, "rhythm_label no debe ser feature"
assert config.CASE_ID_COLUMN not in feature_cols, "case_id no debe ser feature"
assert "beat_type" not in feature_cols, "beat_type no debe ser feature"
assert len(feature_cols) > 0, "No quedaron columnas numéricas válidas"

X      = df[feature_cols].to_numpy(dtype=float)
y      = df[config.TARGET_COLUMN].to_numpy()
groups = df[config.CASE_ID_COLUMN].to_numpy()

print(f"Features seleccionadas ({len(feature_cols)}):")
print(feature_cols)
print(f"\nX shape  : {X.shape}")
print(f"y shape  : {y.shape}")
print(f"NaN en X : {int(np.isnan(X).sum())}  (serán imputados dentro del Pipeline)")
print(f"\nGrupos únicos : {np.unique(groups).shape[0]}")
print(f"Clases únicas : {np.unique(y).tolist()}")

Features seleccionadas (26):
['mean', 'std', 'var', 'min', 'max', 'range', 'median', 'p25', 'p75', 'iqr', 'skew', 'kurtosis', 'energy', 'zero_crossing_rate', 'abs_mean', 'rr_prev', 'rr_next', 'rr_mean_local', 'rr_ratio', 'case_rr_count', 'case_rr_mean', 'case_rr_std', 'case_rr_min', 'case_rr_max', 'case_rr_rmssd', 'case_rr_pnn50']

X shape  : (638690, 26)
y shape  : (638690,)
NaN en X : 1920  (serán imputados dentro del Pipeline)

Grupos únicos : 481
Clases únicas : ['AFIB/AFL', 'AVB', 'N', 'Patterned Atrial Ectopy', 'Patterned Ventricular Ectopy', 'SND', 'SVTA', 'Unclassifiable', 'VT', 'WAP/MAT']


## 6. Split por `case_id` — sin leakage entre pacientes

`make_train_test_group_split_with_coverage` garantiza que todas las clases estén
representadas en train. Se verifica explícitamente que no haya grupos en común entre
train y test.

In [7]:
train_idx, test_idx, split_info = make_train_test_group_split_with_coverage(
    X, y, groups,
    test_size=TEST_SIZE,
    random_state=config.RANDOM_SEED,
)

X_train, X_test           = X[train_idx],      X[test_idx]
y_train, y_test           = y[train_idx],      y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

# Validación estricta: ningún case_id puede estar en ambos conjuntos
train_groups_set = set(groups_train.tolist())
test_groups_set  = set(groups_test.tolist())
overlap = train_groups_set & test_groups_set
assert len(overlap) == 0, f"Leakage detectado: grupos en ambos lados = {overlap}"

print("Validación de leakage: OK (0 grupos en común entre train y test)")
print(f"Grupos en train : {len(train_groups_set)}")
print(f"Grupos en test  : {len(test_groups_set)}")
print(f"Ventanas train  : {len(train_idx):,}")
print(f"Ventanas test   : {len(test_idx):,}")
print(f"\nSplit info: {split_info}")
print(f"\nDistribución de clases en TRAIN:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nDistribución de clases en TEST:")
print(pd.Series(y_test).value_counts().sort_index())

Validación de leakage: OK (0 grupos en común entre train y test)
Grupos en train : 384
Grupos en test  : 97
Ventanas train  : 509,707
Ventanas test   : 128,983

Split info: {'chosen_seed': 43, 'n_classes_covered': 10, 'n_total_classes': 10, 'attempts_tried': 2, 'actual_test_fraction': 0.2019493024785107, 'requested_test_size': 0.2, 'train_groups': [1001, 1018, 1023, 103, 1063, 1072, 1083, 1086, 110, 1110, 1115, 1121, 1127, 114, 1157, 1165, 1191, 12, 1207, 1269, 1276, 1292, 1293, 13, 1314, 1317, 1350, 1367, 1375, 1378, 1398, 1404, 1407, 1415, 146, 1484, 1488, 1523, 158, 1590, 1605, 1607, 1622, 1623, 1626, 1628, 1632, 166, 1665, 1699, 1714, 1730, 1733, 1776, 178, 1798, 1800, 1827, 1828, 1840, 19, 1901, 1903, 1914, 1915, 1957, 1959, 1994, 2008, 2016, 2042, 2054, 2058, 2071, 208, 2103, 2149, 2158, 2161, 2191, 2212, 2213, 2221, 2231, 2246, 2252, 2283, 2296, 230, 2305, 2345, 2348, 2349, 2381, 2397, 2399, 2400, 2424, 2428, 2432, 2447, 2453, 2489, 2495, 2505, 251, 2556, 2558, 257, 2585, 2588, 

## 7. CV splitter interno

Usa `StratifiedGroupKFold` si es viable (sklearn ≥ 1.0); cae a `GroupKFold` si no.
El número de folds se recorta al número de grupos en train cuando sea necesario.

In [8]:
cv, cv_name, n_splits_eff = build_cv_splitter(
    groups_train=groups_train,
    y_train=y_train,
    n_splits=N_SPLITS,
    prefer_stratified=True,
)
print(f"CV splitter     : {cv_name}")
print(f"n_splits efectivo: {n_splits_eff}")

CV splitter     : StratifiedGroupKFold
n_splits efectivo: 3


## 8. Entrenamiento con `RandomizedSearchCV` — un modelo a la vez

Cada modelo se envuelve en `try/except`. Si falla (XGBoost no instalado, memoria insuficiente,
etc.), se registra `status='error'` con el mensaje de error y el resto de modelos continúa.
El test **no se toca** durante la búsqueda: solo se evalúa una vez al final.

> **Nota**: Con `FAST_MODE=True` el entrenamiento tarda ~2-10 min según hardware.
> Para resultados finales, poner `FAST_MODE=False` y re-ejecutar.

In [9]:
model_rows = []

for model_name in MODELS_TO_RUN:

    if model_name not in MODEL_REGISTRY:
        model_rows.append({
            "model": model_name,
            "status": "unknown",
            "error_message": f"Modelo {model_name!r} no está en MODEL_REGISTRY",
        })
        print(f"[{model_name}] status=unknown — no está en MODEL_REGISTRY")
        continue

    spec = MODEL_REGISTRY[model_name]
    logger.info("=== Entrenando: %s ===", model_name)

    try:
        # Verificar dependencias opcionales antes de iniciar la búsqueda
        if model_name == "xgboost":
            try:
                import xgboost  # noqa: F401
            except ImportError as ie:
                raise ImportError(
                    "xgboost no está instalado. Ejecutar: pip install xgboost"
                ) from ie

        res  = run_search_for_model(
            spec=spec,
            X_train=X_train,
            y_train=y_train,
            groups_train=groups_train,
            cv=cv,
            n_iter=N_ITER,
            random_state=config.RANDOM_SEED,
            n_jobs=N_JOBS,
        )
        test = evaluate_on_test(res, X_test, y_test)

        row = {
            "model":              model_name,
            "status":             "ok",
            "n_features":         int(X_train.shape[1]),
            "train_size":         int(len(train_idx)),
            "test_size":          int(len(test_idx)),
            "n_train_groups":     len(train_groups_set),
            "n_test_groups":      len(test_groups_set),
            "best_params":        json.dumps(res["best_params"], default=str),
            "fit_time_seconds":   round(res["fit_seconds"], 2),
            "cv_f1_macro":        res["cv_metrics"].get("cv_f1_macro", float("nan")),
            "test_accuracy":      test.get("test_accuracy",       float("nan")),
            "test_precision_macro": test.get("test_precision_macro", float("nan")),
            "test_recall_macro":  test.get("test_recall_macro",   float("nan")),
            "test_f1_macro":      test.get("test_f1_macro",       float("nan")),
            "test_f1_weighted":   test.get("test_f1_weighted",    float("nan")),
            "error_message":      "",
            # Objetos en memoria — no se serializan en el CSV
            "_best_estimator":    res["best_estimator"],
            "_y_pred_test":       test["y_pred"],
        }
        logger.info("  %s OK: test_f1_macro=%.4f", model_name, row["test_f1_macro"])

    except Exception as exc:
        logger.error("  %s FALLÓ: %s", model_name, exc)
        row = {
            "model":              model_name,
            "status":             "error",
            "n_features":         int(X_train.shape[1]),
            "train_size":         int(len(train_idx)),
            "test_size":          int(len(test_idx)),
            "n_train_groups":     len(train_groups_set),
            "n_test_groups":      len(test_groups_set),
            "best_params":        "{}",
            "fit_time_seconds":   float("nan"),
            "cv_f1_macro":        float("nan"),
            "test_accuracy":      float("nan"),
            "test_precision_macro": float("nan"),
            "test_recall_macro":  float("nan"),
            "test_f1_macro":      float("nan"),
            "test_f1_weighted":   float("nan"),
            "error_message":      f"{type(exc).__name__}: {exc}",
            "_best_estimator":    None,
            "_y_pred_test":       None,
        }

    model_rows.append(row)
    status_str = row["status"]
    extra = (
        f"test_f1_macro={row['test_f1_macro']:.4f}  fit={row['fit_time_seconds']}s"
        if status_str == "ok"
        else row["error_message"][:100]
    )
    print(f"[{model_name}] status={status_str}  {extra}")

2026-05-19 21:04:08 | nb06 | INFO | === Entrenando: linear_svc ===
2026-05-19 21:07:03 | nb06 | INFO |   linear_svc OK: test_f1_macro=0.3439
2026-05-19 21:07:03 | nb06 | INFO | === Entrenando: decision_tree ===


[linear_svc] status=ok  test_f1_macro=0.3439  fit=173.27s


2026-05-19 21:07:42 | nb06 | INFO |   decision_tree OK: test_f1_macro=0.2332
2026-05-19 21:07:42 | nb06 | INFO | === Entrenando: random_forest ===


[decision_tree] status=ok  test_f1_macro=0.2332  fit=38.47s


2026-05-19 21:32:59 | nb06 | INFO |   random_forest OK: test_f1_macro=0.3225
2026-05-19 21:32:59 | nb06 | INFO | === Entrenando: xgboost ===


[random_forest] status=ok  test_f1_macro=0.3225  fit=1514.87s


KeyboardInterrupt: 

## 9. Construcción de `comparison_df`

Se construye explícitamente con columna `status` garantizada. Las columnas `_best_estimator`
y `_y_pred_test` (objetos Python, no serializables) se excluyen del DataFrame para no romper
el guardado a CSV pero se mantienen en `model_rows` para las secciones siguientes.

In [ ]:
# Columnas que van al DataFrame (excluye claves internas con prefijo '_')
REPORT_COLS = [
    "model", "status", "n_features", "train_size", "test_size",
    "n_train_groups", "n_test_groups", "best_params", "fit_time_seconds",
    "cv_f1_macro", "test_accuracy", "test_precision_macro",
    "test_recall_macro", "test_f1_macro", "test_f1_weighted", "error_message",
]

comparison_df = pd.DataFrame([
    {c: row.get(c, float("nan")) for c in REPORT_COLS}
    for row in model_rows
])

print(f"comparison_df shape : {comparison_df.shape}")
print(f"Columnas: {comparison_df.columns.tolist()}")
print()
cols_show = ["model", "status", "test_f1_macro", "test_precision_macro",
             "test_recall_macro", "test_accuracy"]
cols_show = [c for c in cols_show if c in comparison_df.columns]
print(comparison_df[cols_show].round(4).to_string(index=False))

## 10. Comparación de modelos

Celda defensiva: verifica si `comparison_df` está vacío, si existe `status`, y si existe la
métrica primaria. Como `features_baseline` corresponde a una sola configuración de ventana,
no se pivota por `window_seconds`; se muestra una tabla simple ordenada por `test_f1_macro`.

In [ ]:
PRIMARY_METRIC = f"test_{PRIMARY_SCORING}"  # 'test_f1_macro'

if comparison_df.empty:
    print("AVISO: comparison_df está vacío. No hay resultados que mostrar.")

elif "status" not in comparison_df.columns:
    print("AVISO: comparison_df no tiene columna 'status'.")
    print("Columnas disponibles:", comparison_df.columns.tolist())

else:
    ok = comparison_df.loc[comparison_df["status"] == "ok"].copy()

    if ok.empty:
        print("AVISO: Ningún modelo completó con status='ok'. Tabla de errores:")
        display(comparison_df[["model", "status", "error_message"]])

    else:
        # Resolver métrica primaria a mostrar
        if PRIMARY_METRIC not in ok.columns:
            PRIMARY_METRIC = "test_f1_macro"
            print(f"[AVISO] Usando '{PRIMARY_METRIC}' como métrica principal.")

        # Una sola ventana → tabla simple (no pivotear por window_seconds)
        display_cols = [
            "model", "test_f1_macro", "test_precision_macro",
            "test_recall_macro", "test_accuracy",
            "cv_f1_macro", "fit_time_seconds",
        ]
        display_cols = [c for c in display_cols if c in ok.columns]

        print(f"Comparación de modelos — ordenada por {PRIMARY_METRIC}:\n")
        display(
            ok[display_cols]
            .sort_values(PRIMARY_METRIC, ascending=False)
            .reset_index(drop=True)
            .round(4)
        )

        # Mostrar modelos con error si los hay
        err = comparison_df.loc[comparison_df["status"] != "ok"]
        if not err.empty:
            print("\nModelos que fallaron:")
            display(err[["model", "status", "error_message"]])

## 11. Mejor modelo global

Se selecciona el modelo con mayor `test_f1_macro`. Se reporta el classification report
completo y la tabla por clase con soporte.

In [ ]:
ok_rows = [
    r for r in model_rows
    if r.get("status") == "ok" and r.get("_y_pred_test") is not None
]

if not ok_rows:
    print("Sin ganadores válidos (status='ok'). Revisa los errores en la tabla anterior.")
    winner_row    = None
    winner_name   = None
    y_pred_winner = None

else:
    winner_row    = max(ok_rows, key=lambda r: r.get("test_f1_macro", -1.0))
    winner_name   = winner_row["model"]
    y_pred_winner = winner_row["_y_pred_test"]

    print("=" * 60)
    print(f"MEJOR MODELO: {winner_name}")
    print("=" * 60)
    print(f"  test_f1_macro        : {winner_row['test_f1_macro']:.4f}")
    print(f"  test_precision_macro : {winner_row['test_precision_macro']:.4f}")
    print(f"  test_recall_macro    : {winner_row['test_recall_macro']:.4f}")
    print(f"  test_accuracy        : {winner_row['test_accuracy']:.4f}")
    print(f"  cv_f1_macro (train)  : {winner_row['cv_f1_macro']:.4f}")
    print(f"  fit_time_seconds     : {winner_row['fit_time_seconds']:.1f}s")
    print(f"  Mejores hiperparámetros: {winner_row['best_params']}")
    print()
    print("Classification report completo (test):")
    print(classification_report(y_test, y_pred_winner, zero_division=0))
    print()
    print("Reporte por clase (tabla):")
    display(per_class_report(y_test, y_pred_winner).round(3))
    print()
    print("Matriz de confusión (absoluta con totales de fila y columna):")
    display(confusion_matrix_with_totals(y_test, y_pred_winner))

## 12. Matriz de confusión (visual)

Conteos absolutos. Filas = clase real; columnas = clase predicha.

In [ ]:
if winner_row is None:
    print("Sin ganador válido; nada que graficar.")
else:
    labels = sorted(
        set(pd.Series(y_test).unique()) | set(pd.Series(y_pred_winner).unique()),
        key=str,
    )
    cm = confusion_matrix(y_test, y_pred_winner, labels=labels)

    fig, ax = plt.subplots(figsize=(1.0 + len(labels), 0.8 + 0.8 * len(labels)))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=labels, yticklabels=labels,
        cbar=False, ax=ax,
    )
    ax.set_title(f"Matriz de confusión — {winner_name}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout()

    config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    cm_path = config.FIGURES_DIR / "best_model_confusion_matrix.png"
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Guardada: {cm_path}")

## 13. Importancia de variables del mejor modelo

- Si el modelo tiene `feature_importances_` (Random Forest, XGBoost, Decision Tree): se usa directamente.
- Si tiene `coef_` (LinearSVC, LogReg): se calcula la norma L2 de los coeficientes por feature.
- Si no tiene ninguno: se informa claramente.

In [ ]:
if winner_row is None:
    print("Sin ganador; no hay importancia de variables.")

else:
    best_estimator = winner_row["_best_estimator"]
    clf = best_estimator.named_steps["clf"]

    importances = None
    imp_label   = None

    if hasattr(clf, "feature_importances_"):
        importances = clf.feature_importances_
        imp_label   = "feature_importances_"
    elif hasattr(clf, "coef_"):
        coef = np.asarray(clf.coef_)
        importances = np.linalg.norm(coef, axis=0) if coef.ndim > 1 else np.abs(coef.ravel())
        imp_label   = "|coef_| (norma L2 por feature)"

    if importances is not None:
        imp_df = (
            pd.DataFrame({"feature": feature_cols, "importance": importances})
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )
        print(f"Fuente de importancia : {imp_label}")
        print(f"\nTop 10 features — {winner_name}:")
        print(imp_df.head(10).to_string(index=False))

        config.TABLES_DIR.mkdir(parents=True, exist_ok=True)
        imp_path = config.TABLES_DIR / "best_model_feature_importance.csv"
        imp_df.head(10).to_csv(imp_path, index=False)
        print(f"\nGuardado: {imp_path}")

        top10 = imp_df.head(10)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top10["feature"][::-1], top10["importance"][::-1])
        ax.set_xlabel("Importancia")
        ax.set_title(f"Top 10 features — {winner_name}")
        plt.tight_layout()
        plt.show()

    else:
        print(
            f"El modelo {winner_name} no expone importancia de variables directa.\n"
            "(No tiene `feature_importances_` ni `coef_`)"
        )

## 14. Guardado de reportes

Archivos generados:
- `reports/tables/model_comparison.csv` — tabla comparativa de todos los modelos.
- `reports/tables/best_model_classification_report.csv` — reporte por clase del ganador.
- `reports/tables/best_model_feature_importance.csv` — top 10 features (si aplica).
- `reports/figures/best_model_confusion_matrix.png` — matriz de confusión visual.

In [ ]:
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Tabla comparativa de modelos (sin columnas de objetos)
comp_path = config.TABLES_DIR / "model_comparison.csv"
comparison_df.to_csv(comp_path, index=False)
print(f"Guardado: {comp_path}")

# Classification report del mejor modelo
if winner_row is not None:
    rep_dict = classification_report(
        y_test, y_pred_winner, output_dict=True, zero_division=0
    )
    rep_df   = pd.DataFrame(rep_dict).T
    rep_path = config.TABLES_DIR / "best_model_classification_report.csv"
    rep_df.to_csv(rep_path)
    print(f"Guardado: {rep_path}")
else:
    print("Sin ganador: no se guardó classification_report.")

print("\nResumen de archivos generados:")
for p in [
    config.TABLES_DIR / "model_comparison.csv",
    config.TABLES_DIR / "best_model_classification_report.csv",
    config.TABLES_DIR / "best_model_feature_importance.csv",
    config.FIGURES_DIR / "best_model_confusion_matrix.png",
]:
    status_icon = "OK" if p.exists() else "--"
    print(f"  [{status_icon}] {p}")

## 15. Notas metodológicas

- El notebook usa `features_baseline.parquet` generado en `04_windowing_and_feature_engineering.ipynb`.
  Si en el futuro se generan parquets con distintas duraciones de ventana, la estructura del notebook
  sigue siendo válida — solo cambiar el archivo de entrada en la sección 4.
- El split y CV están anclados a `case_id`. Ningún paciente aparece en train y test simultáneamente.
- `beat_type` está bloqueado por `assert_no_forbidden_features`; no puede entrar como predictor.
- **Para resultados finales**: poner `FAST_MODE = False` en la sección 2 y re-ejecutar el notebook entero.
- Los reportes finales se guardan en `reports/tables/` y `reports/figures/`.
- Antes de interpretar las métricas, revisar la sección 6: si una clase aparece en muy pocos
  `case_id` únicos, puede que quede fuera de train o test en algún split, haciendo su
  recall/F1 no interpretable como métrica estable.